# Transfer learning (ResNet-18) с PyTorch и MLflow

In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models, datasets
import mlflow
import mlflow.pytorch
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

## Настройка

In [9]:
DATA_DIR = Path("../data/processed")
TEST_DIR = Path("../data/raw/seg_test/seg_test")
BATCH_SIZE = 32
NUM_EPOCHS_FROZEN = 10
NUM_EPOCHS_FINETUNE = 10
LR_FROZEN = 1e-3
LR_FINETUNE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## Трансформации

In [10]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Датасеты и загрузчики

In [11]:
train_dataset = datasets.ImageFolder(root=DATA_DIR / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(root=DATA_DIR / "val", transform=val_test_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

classes = train_dataset.classes
print("Classes:", classes)

Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


## Модель ResNet-18

In [12]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))
model = model.to(DEVICE)

c:\Users\User\source\repos_code\aie-DmchFast\project\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\User\source\repos_code\aie-DmchFast\project\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# Заморозка всех слоёв, кроме последнего


In [13]:
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LR_FROZEN)

## Обучение замороженной части

In [ ]:
mlflow.set_experiment("Intel_Image_Classification")
with mlflow.start_run(run_name="ResNet18_frozen"):
    for epoch in range(NUM_EPOCHS_FROZEN):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Валидация
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        acc = accuracy_score(all_labels, all_preds)
        
        mlflow.log_metric("loss", running_loss/len(train_loader), step=epoch)
        mlflow.log_metric("val_accuracy", acc, step=epoch)
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Val Acc: {acc:.4f}")
    
    # Сохраняем модель
    mlflow.pytorch.log_model(model, "model_frozen")
    torch.save(model.state_dict(), "../artifacts/resnet18_frozen.pth")

Epoch 1: 100%|██████████| 351/351 [03:17<00:00,  1.77it/s]


Epoch 1, Loss: 0.6455, Val Acc: 0.8886


Epoch 2: 100%|██████████| 351/351 [03:23<00:00,  1.73it/s]


Epoch 2, Loss: 0.4435, Val Acc: 0.8922


Epoch 3: 100%|██████████| 351/351 [03:24<00:00,  1.72it/s]


...

Epoch 8, Loss: 0.4103, Val Acc: 0.9036


Epoch 9: 100%|██████████| 351/351 [03:21<00:00,  1.74it/s]


Epoch 9, Loss: 0.4030, Val Acc: 0.9053


Epoch 10: 100%|██████████| 351/351 [03:21<00:00,  1.74it/s]
2026/05/20 02:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 02:41:19 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Epoch 10, Loss: 0.4146, Val Acc: 0.9028


## Разморозка всех слоёв и дообучение

In [ ]:
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=LR_FINETUNE)

with mlflow.start_run(run_name="ResNet18_finetune"):
    for epoch in range(NUM_EPOCHS_FINETUNE):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Валидация
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        acc = accuracy_score(all_labels, all_preds)
        
        mlflow.log_metric("loss", running_loss/len(train_loader), step=epoch)
        mlflow.log_metric("val_accuracy", acc, step=epoch)
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Val Acc: {acc:.4f}")
    
    # Сохраняем финальную модель
    mlflow.pytorch.log_model(model, "model_finetuned")
    torch.save(model.state_dict(), "../artifacts/best_model.pth")

Epoch 1: 100%|██████████| 351/351 [08:54<00:00,  1.52s/it]


Epoch 1, Loss: 0.4230, Val Acc: 0.9043


Epoch 2: 100%|██████████| 351/351 [08:52<00:00,  1.52s/it]


Epoch 2, Loss: 0.3445, Val Acc: 0.9302


Epoch 3: 100%|██████████| 351/351 [08:53<00:00,  1.52s/it]


...

Epoch 8, Loss: 0.2346, Val Acc: 0.9224


Epoch 9: 100%|██████████| 351/351 [08:52<00:00,  1.52s/it]


Epoch 9, Loss: 0.2279, Val Acc: 0.9253


Epoch 10: 100%|██████████| 351/351 [08:55<00:00,  1.53s/it]
2026/05/20 04:18:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 04:18:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Epoch 10, Loss: 0.2221, Val Acc: 0.9363


## Оценка на тестовой выборке

In [16]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(all_labels, all_preds, target_names=classes))

Test accuracy: 0.9270
              precision    recall  f1-score   support

   buildings       0.96      0.88      0.91       437
      forest       1.00      0.99      0.99       474
     glacier       0.93      0.85      0.89       553
    mountain       0.89      0.90      0.90       525
         sea       0.90      0.99      0.94       510
      street       0.90      0.96      0.93       501

    accuracy                           0.93      3000
   macro avg       0.93      0.93      0.93      3000
weighted avg       0.93      0.93      0.93      3000

